In [1]:
# to help with importing app files
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.path.abspath(''), '..')))

# New Route Worksheet

This notebook runs through how to create a new route within this dummy app. It includes all the code you need. You simply have to save the code in the correct locations, run the app, and see if it works. Together, **we're going to build a route that determines when an individual's birthday is next year**. It's a pretty useless feature, but very easy to program. 

Here are the steps to be completed:

0. Check the app works in your environment
1. Create a new branch
2. Plan your new route
3. Create the business logic 

    a. Program the utils

    b. Unit test your utils

4. Create the service (the orchestrator of your business logic)

    a. Create the pydantic models

    b. Program the orchestrator function

    c. Integration test your service

5. Create the route

    a. Integration test the route
    
6. Demo your new route

## 0. Check the app works in your environment
After having pulled the code and setup a virtual environment, run the app locally to check all necessary dependencies (e.g., Flask) have been installed. The app can be run via the terminal by using `python -B app.py`. Make sure you are in the root directory. With the app running, try executing the following cell. Do you get an error? Or do you get a success response?

In [2]:
import requests
from pprint import pprint
from datetime import datetime

# user information
name = "Alice"
today = datetime.today()
birthday =today.strftime("%Y-%m-%d")

# request payload
body = {
    "name":name,
    "date":birthday
}

# make the request
url = f'http://localhost:5000/api/v1/birthdayhaiku'
x = requests.post(url, json=body)

# pretty print the result
pprint(x.json())

{'failures': None,
 'inferenceMetadata': {'apiVersion': '1',
                       'requestId': '1e686bda-0e60-4f1c-9724-ffcde741aa32',
                       'route': '/api/v1/birthdayhaiku',
                       'status': 'SUCCESS'},
 'result': {'haiku': 'Bright candles glimmer\n'
                     'Alice laughs among the stars\n'
                     'Joy blooms with each breath',
            'isBirthday': True}}


## 1. Create a new branch
With the app working in your environment, it's time to make changes. **BEFORE YOU CHANGE ANYTHING**, let's create a new branch. This way, we can rest assured knowing that we won't be changing any working code that someone else might be using. 

To create a branch, in the terminal run `git checkout -b feat/<new branch name>`. Remember to substitute "new branch name" with something that communicates to others what new feature your branch includes. For example, we're building a birthday next year feature, so we could use `feat/next-year`. The command `-b` instructs the terminal to create a new branch rather than switching to an existing one. 

## 2. Plan your new route
To help us determine what we're about to program, we first want to think about the logic of our route. We need to answer questions like:
1. What is it we want to build?
2. How will I use Python, ML, AI, etc. to generate the content being requested?
3. What data do I need from the user to achieve this?
4. How will I return a response to them?

In our use case:
1. We want to get the user's birthday next year. To be fun, we could also include a separate value that specifies the day of the week.
2. We don't need AI. Python's `datetime` module can easily store and manipulate date objects and `dateutils.relativedelta` can find the date of their birthday next year. The `calendar` package can get the day of the week in English.
3. We will need the user's birthday as an input. For ease, this can be a date in the format `%Y-%m-%d`. 
4. The response will include two fields. Firstly, it'll include a date. If they're sending a date as a string, it makes sense to return it as a string too. Secondly, we'll have the day of the week as a string.

This information will help us with building our business logic. 

## 3. Create the business logic
#### a. Program the utils
Backend utils will be used to generate the requested content. Writing "modular code" (functions that each perform separate roles to achieve a greater task) will make it easier to test our code later, to ensure everything is performing their individual part as expected. 

While writing your backend code, you want to consider "how can people use my functions incorrectly?" and "what could possibly go wrong?". For example, what if somebody parses the wrong data type? Even though we use Pydantic to enforce data structures, someone in the future may use our util in a different context. And what if an external resource can't be accessed? A docstring can be included to help others with understanding a function's purpose.

In the below cell are the backend utils. **Copy and paste these imports and utils into `app/v1/utils/birthday_next_year.py`**.

In [3]:
from datetime import datetime
from dateutil.relativedelta import relativedelta
import calendar
from typing import Union

def check_datetime_input(date: Union[str, datetime]) -> datetime:
    """
    Check the datatype of an inputted date and convert to datetime
    Parameters
    ----------
    date : Union[str, datetime]
        Either a string in the format "%Y-%m-%d" or a datetime object

    Returns
    -------
    datetime
        The correctly formatted datetime object
    """
    # what if someone doesn't parse a datetime (Python doesn't automatically enforce datatypes in arguments)
    # if it's a string, try to convert it
    if not isinstance(date, datetime):
        try:
            datetime_obj = datetime.strptime(date, "%Y-%m-%d")
        except:
            raise Exception("Datetime string is not in the correct date format. Expected '%Y-%m-%d'")
        
    # if it's already a datetime, nothing we need to do
    elif isinstance(date, datetime):
        datetime_obj = date

    # otherwise, can't determine date
    else:
        raise Exception("Inputted date is not string or datetime.")
    
    return datetime_obj

def calculate_birthday_next_year(birthday: datetime) -> datetime:
    """
    Given a person's birthday, calculate what day and date their birthday is next year
    Parameters
    ----------
    birthday : datetime
        The date of someone's birthday

    Returns
    -------
    datetime
        The person's birthday next year
    """
    # check input structure
    birthday_datetime = check_datetime_input(birthday)

    # construct birthday next year (disregard year input)
    date_next_year = datetime.today() + relativedelta(years=1) 
    next_year = date_next_year.year
    birthday_day = birthday_datetime.day
    birthday_month = birthday_datetime.month
    birthday_next_year = datetime(next_year, birthday_month, birthday_day)

    return birthday_next_year

def get_date_day_of_week(date: datetime) -> str:
    """
    Given a date, use the calendar package to get the day of the week
    Parameters
    ----------
    date : datetime
        The date to be processed

    Returns
    -------
    str
        The day of the week
    """    
    # check input structure
    datetime_obj = check_datetime_input(date)
    dow = calendar.day_name[datetime_obj.weekday()]

    return dow

Here's a demonstration of them working together (and provides a hint to how the service will be structured). 

In [4]:
# string date input
input_date = "1998-08-24"
next_birthday = calculate_birthday_next_year(input_date)
dow = get_date_day_of_week(next_birthday)
print(f"Your next birthday is on {next_birthday.strftime("%d/%m/%Y")} which happens to be a {dow}.")

Your next birthday is on 24/08/2026 which happens to be a Monday.


In [5]:
# datetime date input
input_date = datetime(1998, 8, 24)
next_birthday = calculate_birthday_next_year(input_date)
dow = get_date_day_of_week(next_birthday)
print(f"Your next birthday is on {next_birthday.strftime("%d/%m/%Y")} which happens to be a {dow}.")

Your next birthday is on 24/08/2026 which happens to be a Monday.


#### b. Unit test the utils
This is a good time to make sure all the functions are behaving as expected. This is quite a time consuming task, so we haven't written **all** the tests that would be required, but here's some examples. We're using the AAA framework: setup the test, use the function we're testing, and assert that everything behaved as expected. You would save all your tests in `tests/v1/unit/test_birthday_next_year.py` and you would use the command `pytest` in the terminal to run all tests at once. 

In [6]:
import pytest

def test_check_datetime_input_happy_path_datetime_input():
    """
    GIVEN a datetime object 
    WHEN check_datetime_input is called
    THEN the input is returned unchanged
    """
    # --- 1. Arrange ---
    input = datetime(2000, 1, 1)
    # --- 2. Act ---
    output = check_datetime_input(input)
    # --- 3. Assert ---
    assert input == output

def test_check_datetime_input_happy_path_str_input():
    """
    GIVEN a string object of a datetime in the expected format
    WHEN check_datetime_input is called
    THEN the input is converted to datetime
    """
    # --- 1. Arrange ---
    input = "2000-01-01"
    # --- 2. Act ---
    output = check_datetime_input(input)
    # --- 3. Assert ---
    assert datetime.strptime(input, "%Y-%m-%d") == output

def test_check_datetime_input_bad_str_input():
    """
    GIVEN a string object of a datetime in an unexpected format
    WHEN check_datetime_input is called
    THEN the input is converted to datetime
    """
    # --- 1. Arrange ---
    input = "01/01/2000"
    # --- 2. Act & Assert ---
    with pytest.raises(Exception) as context:
        check_datetime_input(input)

# this is not the way to run tests, we'd use the command `pytest` in the terminal
# this is just to demo that they run :) 
test_check_datetime_input_happy_path_datetime_input()
test_check_datetime_input_happy_path_str_input()
test_check_datetime_input_bad_str_input()

## 4. Create the service
With the utils ready, we need to build our orchestrator function that calls all utils in the correct order. This function is responsible for handling the business logic of our route. 

#### a. Pydantic Models
The service will require pydantic models to control the structure of inputs and outputs. If someone makes a dodgy request (e.g. don't include a date), Pydantic can catch it for us.

**Copy and paste these models into `app/v1/schemas.py`**

In [7]:
# these imports are NOT needed in the schemas.py file, they are already there :)
from pydantic import (    
    BaseModel,
    field_serializer,
    Field,
    ConfigDict
)
from app.v1.schemas import DateRequestModel, ServiceResult

# the service request model can inherit from the existing DateRequestModel
class NextYearRequestModel(DateRequestModel):
    pass

# the data that the service will be creating
class NextYearResponseModel(BaseModel):
    next_year_date: datetime = Field(
        ...,
        # convert snake case to camel case
        alias="nextYearDate",
        description="The requesters birthday next year"
    )
    day_of_week: str = Field(
        ...,
        alias="dayOfWeek",
        description="Day of week of next year's birthday"
    )

    # when converting a NextYearResponseModel object to a dictionary, this serializer will convert the datetime object to a string (so clever!)
    @field_serializer("next_year_date", mode="plain")
    @classmethod
    def format_next_year_date(cls, date):
        return date.strftime("%Y-%m-%d")
    
    model_config = ConfigDict(
        populate_by_name=True,
    )

# the structure of the output from the service - this ensures all services return the same structure
NextYearServiceResult = ServiceResult[NextYearResponseModel]  

And here's some examples of their usage:

In [8]:
example_request = NextYearRequestModel(
    date="1998-08-24"
)

print("The inputted request object (string has been converted to datetime):")
print(example_request)

example_response = NextYearResponseModel(
    next_year_date=datetime(2026, 8, 24),
    day_of_week="Monday"
)

print("\nThe serialized service-specific content (datetime has been converted to string):")
print(example_response.model_dump(by_alias=True))

example_service_result = NextYearServiceResult(
    data=example_response
)

print("\nThe final service output:")
print(example_service_result.model_dump(by_alias=True))

The inputted request object (string has been converted to datetime):
date=datetime.datetime(1998, 8, 24, 0, 0)

The serialized service-specific content (datetime has been converted to string):
{'nextYearDate': '2026-08-24', 'dayOfWeek': 'Monday'}

The final service output:
{'data': {'nextYearDate': '2026-08-24', 'dayOfWeek': 'Monday'}}


#### b. Program the orchestrator function
This is the function that the route will be calling to generate the content. It utilises the pydantic models and utils functions.

**Copy and paste the below code into `app/v1/services.py`**.

In [9]:
# to check the structure of requests are correct
from pydantic import validate_call

# service-specific utils
# (these imports will only work if the above utils have been copied and pasted into `birthday_next_year.py`)
from app.v1.utils.birthday_next_year import calculate_birthday_next_year, get_date_day_of_week

# request and response models for all services 
# (these imports will only work if the above models have been copied and pasted into `schemas.py`)
from app.v1.schemas import NextYearRequestModel, NextYearResponseModel, NextYearServiceResult

# this function decorator will ensure that all inputs into our service are instances of NextYearRequestModel
@validate_call
def next_year_service(request: NextYearRequestModel) -> NextYearServiceResult:
    """
    Service to calculate the date and day-of-week of a user's birthday
    """
    
    # 1. get datetime of birthday next year
    next_birthday = calculate_birthday_next_year(request.date)

    # 2. Get day-of-week of next year's birthday
    dow = get_date_day_of_week(next_birthday)    

    # 3. Generate the response
    next_year_result = NextYearResponseModel(
        next_year_date=next_birthday,
        day_of_week=dow
    )

    response = NextYearServiceResult(
        data=next_year_result
    )

    return response

And here's its example usage:

In [10]:
example_request = NextYearRequestModel(
    date="1998-08-24"
)

# use service
next_birthday = next_year_service(example_request)
# extract information from the response
# (all service responses will firstly have the key "data")
next_birthday_info = next_birthday.data.model_dump(by_alias=True)
print(f"Your next birthday is on {next_birthday_info["nextYearDate"]} which happens to be a {next_birthday_info["dayOfWeek"]}.")

Your next birthday is on 2026-08-24 which happens to be a Monday.


#### c. Integration test the service
We need to test different scenarios, to check the output from the service is as expected. They would be considered integration tests as they test the services ability to use several utils together to create an output. These would live in `tests/v1/integration/test_next_year_service.py`. Here's an example test:

In [11]:
def test_next_year_service_dict_input():
    """
    GIVEN a correctly formatted dict object that conforms to NextYearRequestModel
    WHEN next_year_service is called
    THEN the date next year and day-of-week are calculated
    """
    # --- 1. Arrange ---
    input = {
        "date": "2000-01-01"
    }
    # --- 2. Act ---
    result = next_year_service(input)
    # --- 3. Assert ---
    # structure is correct
    assert isinstance(result, NextYearServiceResult)
    # info is correct
    result_dict = result.data.model_dump(by_alias=True)
    # this will only assert true if ran before 2026
    assert result_dict["nextYearDate"] == "2026-01-01"
    assert result_dict["dayOfWeek"] == "Thursday"
    
# this is not the way to run tests, we'd use the command `pytest` in the terminal
# this is just to demo that they run :) 
test_next_year_service_dict_input()

## 5. Create the route
This is the easy bit! We need to create the Flask endpoint using the existing route decorator - this file handles all the Flask logic such as reading the request and returning a response. 

**Copy and paste the below into `app/v1/routes.py`**. This code won't work in the notebook as it relies on the app factory (`app/__init__.py`), but that's OK, we know how to run the app in the terminal!

In [ ]:
# (this import will only work if the models has been copied and pasted into `schemas.py`)
from app.v1.schemas import NextYearRequestModel, NextYearServiceResult
# (this import will only work if the service has been copied and pasted into `services.py`)
from app.v1.services import next_year_service

# define the endpoint name and assign it to the v1 blueprint
@v1_bp.route(f"/birthdaynextyear", methods=["POST"])
# use the route decorator, specifying the expected data structures for our service
@api_route(
    http_request_model=NextYearRequestModel, 
    service_result_model=NextYearServiceResult,
)
def birthday_next_year(validated_input: NextYearRequestModel):
    """
    API Endpoint for calculating a birthday next year
    """
    return next_year_service(validated_input)

## 6. Demo your new route 
With everything in the right place and tested, let's get the app running and try a request!

In [12]:
import requests
from pprint import pprint

# user information
birthday = "1998-08-24"

# request payload
body = {
    "date":birthday
}

# make the request
url = f'http://localhost:5000/api/v1/birthdaynextyear'
x = requests.post(url, json=body)

# pretty print the result
pprint(x.json())

{'failures': None,
 'inferenceMetadata': {'apiVersion': '1',
                       'requestId': 'b0695103-c8d2-45ba-a988-4538e294394f',
                       'route': '/api/v1/birthdaynextyear',
                       'status': 'SUCCESS'},
 'result': {'dayOfWeek': 'Monday', 'nextYearDate': '2026-08-24'}}


Congrats! You have successfully created a new route in our dummy app! 🎉 Try designing a different route (maybe an age calculator) and have a go writing the code yourself.